# Common Configuration Loader

**Purpose:** Centralized configuration management for all pipeline notebooks.

**Usage:**
```python
%run "./common/common_config_loader"

config = get_config()
raw_path = config.RAW_LANDING_PATH
bronze_table = config.get_full_table_name("bronze", "orders")
```

**Features:**
* Environment-specific configurations
* Centralized path definitions
* Table name builders
* Schema definitions
* Easy configuration updates

In [0]:
from typing import Optional, Dict, Any
from dataclasses import dataclass

# ============================================================================
# PIPELINE CONFIGURATION
# ============================================================================

@dataclass
class PipelineConfig:
    """
    Centralized configuration for data pipeline.
    
    All paths, catalogs, schemas, and environment-specific settings
    are defined here to avoid duplication across notebooks.
    """
    
    # ========================================================================
    # UNITY CATALOG CONFIGURATION
    # ========================================================================
    
    # Catalogs
    CATALOG: str = "workspace"
    
    # Schemas
    BRONZE_SCHEMA: str = "bronze"
    SILVER_SCHEMA: str = "silver"
    GOLD_SCHEMA: str = "gold"
    ECOMMERCE_SCHEMA: str = "ecommerce"
    QUARANTINE_SCHEMA: str = "quarantine"
    
    # ========================================================================
    # VOLUME PATHS
    # ========================================================================
    
    # Raw landing zone for ingestion
    RAW_LANDING_PATH: str = "/Volumes/workspace/ecommerce/raw_landing"
    
    # Archive path for processed files
    ARCHIVE_PATH: str = "/Volumes/workspace/ecommerce/archive"
    
    # Temporary staging path
    STAGING_PATH: str = "/Volumes/workspace/ecommerce/staging"
    
    # ========================================================================
    # CONTROL TABLE NAMES
    # ========================================================================
    
    FILE_CHANGE_TRACKER_TABLE: str = "file_change_tracker"
    FILES_TO_PROCESS_TABLE: str = "files_to_process"
    PIPELINE_METADATA_TABLE: str = "pipeline_metadata"
    
    # ========================================================================
    # PROCESSING CONFIGURATION
    # ========================================================================
    
    # Batch size for incremental processing
    DEFAULT_BATCH_SIZE: int = 1000
    
    # Timeout settings (seconds)
    HTTP_TIMEOUT: int = 30
    HTTP_RETRY_COUNT: int = 3
    
    # Date formats
    DATE_FORMAT: str = "%Y-%m-%d"
    DATETIME_FORMAT: str = "%Y-%m-%d %H:%M:%S"
    
    # ========================================================================
    # HELPER METHODS
    # ========================================================================
    
    def get_full_table_name(self, schema: str, table: str) -> str:
        """
        Get fully qualified table name.
        
        Args:
            schema: Schema name (bronze, silver, gold, etc.)
            table: Table name
        
        Returns:
            Fully qualified table name: catalog.schema.table
        
        Example:
            config.get_full_table_name("bronze", "orders")
            # Returns: "workspace.bronze.orders"
        """
        return f"{self.CATALOG}.{schema}.{table}"
    
    def get_bronze_table(self, table: str) -> str:
        """Get fully qualified Bronze table name."""
        return self.get_full_table_name(self.BRONZE_SCHEMA, table)
    
    def get_silver_table(self, table: str) -> str:
        """Get fully qualified Silver table name."""
        return self.get_full_table_name(self.SILVER_SCHEMA, table)
    
    def get_gold_table(self, table: str) -> str:
        """Get fully qualified Gold table name."""
        return self.get_full_table_name(self.GOLD_SCHEMA, table)
    
    def get_control_table(self, table: str) -> str:
        """Get fully qualified control table name (ecommerce schema)."""
        return self.get_full_table_name(self.ECOMMERCE_SCHEMA, table)
    
    def get_quarantine_table(self, table: str) -> str:
        """Get fully qualified quarantine table name."""
        return self.get_full_table_name(self.QUARANTINE_SCHEMA, table)

In [0]:
# ============================================================================
# GLOBAL CONFIGURATION INSTANCE
# ============================================================================

# Singleton configuration instance
_config_instance: Optional[PipelineConfig] = None

def get_config() -> PipelineConfig:
    """
    Get global configuration instance (singleton).
    
    Returns:
        PipelineConfig instance
    
    Example:
        config = get_config()
        table_name = config.get_bronze_table("orders")
    """
    global _config_instance
    
    if _config_instance is None:
        _config_instance = PipelineConfig()
    
    return _config_instance

def update_config(**kwargs) -> None:
    """
    Update configuration values.
    
    Args:
        **kwargs: Configuration key-value pairs to update
    
    Example:
        update_config(CATALOG="dev_workspace", HTTP_TIMEOUT=60)
    """
    config = get_config()
    
    for key, value in kwargs.items():
        if hasattr(config, key):
            setattr(config, key, value)
        else:
            raise ValueError(f"Unknown configuration key: {key}")

In [0]:
def display_config() -> None:
    """
    Display current configuration in a readable format.
    
    Useful for debugging and documentation.
    
    Example:
        display_config()
    """
    config = get_config()
    
    print("="*80)
    print("PIPELINE CONFIGURATION".center(80))
    print("="*80)
    
    print("\nUnity Catalog:")
    print(f"  Catalog: {config.CATALOG}")
    print(f"  Bronze Schema: {config.BRONZE_SCHEMA}")
    print(f"  Silver Schema: {config.SILVER_SCHEMA}")
    print(f"  Gold Schema: {config.GOLD_SCHEMA}")
    print(f"  Ecommerce Schema: {config.ECOMMERCE_SCHEMA}")
    print(f"  Quarantine Schema: {config.QUARANTINE_SCHEMA}")
    
    print("\nVolume Paths:")
    print(f"  Raw Landing: {config.RAW_LANDING_PATH}")
    print(f"  Archive: {config.ARCHIVE_PATH}")
    print(f"  Staging: {config.STAGING_PATH}")
    
    print("\nControl Tables:")
    print(f"  File Tracker: {config.get_control_table(config.FILE_CHANGE_TRACKER_TABLE)}")
    print(f"  Files to Process: {config.get_control_table(config.FILES_TO_PROCESS_TABLE)}")
    print(f"  Pipeline Metadata: {config.get_control_table(config.PIPELINE_METADATA_TABLE)}")
    
    print("\nProcessing Settings:")
    print(f"  Default Batch Size: {config.DEFAULT_BATCH_SIZE:,}")
    print(f"  HTTP Timeout: {config.HTTP_TIMEOUT}s")
    print(f"  HTTP Retry Count: {config.HTTP_RETRY_COUNT}")
    
    print("="*80)